# LNO2d - the Low-rank Neural Operator

*Walkthrough notebook for `neural_operators/models/lno.py`.*

The LNO (paper Section 4.2) takes a different route to a cheap integral: make the kernel **low rank**,

$$\kappa(x,y)\approx\sum_{r=1}^{R}\phi_r(x)\,\psi_r(y)^\top .$$

Then the order of sum and integral can be swapped:

$$(\mathcal Kv)(x)=\sum_r\phi_r(x)\underbrace{\int\psi_r(y)^\top v(y)\,dy}_{s_r\ \text{(one number per }r)} .$$

Computing the $R$ numbers $s_r$ costs $O(RJ)$ and spreading them back with $\phi_r(x)$ costs another $O(RJ)$ -
**linear** in the number of points. The price: a rank-$R$ kernel can only represent $R$ "modes" of interaction,
much like a truncated SVD.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Learned basis functions

One small MLP maps a coordinate to `R` vectors of dimension `d` - these are the $\phi_r(x)$ (or $\psi_r(y)$).

In [ ]:
class BasisMLP(nn.Module):
    """Maps a coordinate x in R^{coord_dim} to R basis vectors, each of dimension d,
    i.e. output shape (N, R, d)."""

    def __init__(self, coord_dim, rank, d, hidden=64):
        super().__init__()
        self.rank = rank
        self.d = d
        self.net = nn.Sequential(
            nn.Linear(coord_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, rank * d),
        )

    def forward(self, coords):
        out = self.net(coords)  # (N, rank*d)
        return out.view(-1, self.rank, self.d)

## One LNO layer

Two `BasisMLP`s give $\phi$ (output side) and $\psi$ (input side). In `forward`:

- `s = einsum("nrd,bnd->br", psi, v) / n_points` is the Monte-Carlo estimate of $\int\psi_r\cdot v$ -
  *one number per basis function*, collapsing the whole domain.
- `integral_term = einsum("nrd,br->bnd", phi, s)` spreads it back to every point.
- the local linear term $W_t v$ is added as in every neural operator.

In [ ]:
class LNOLayer(nn.Module):
    def __init__(self, width, rank=16, coord_dim=2, hidden=64):
        super().__init__()
        self.phi_net = BasisMLP(coord_dim, rank, width, hidden=hidden)  # phi_r(x), output-side basis
        self.psi_net = BasisMLP(coord_dim, rank, width, hidden=hidden)  # psi_r(y), input-side basis
        self.local_linear = nn.Linear(width, width)  # W_t, the local term in Eq. (10)

    def forward(self, v, coords):
        # v: (batch, N, width), coords: (N, coord_dim)
        n_points = coords.shape[0]
        phi = self.phi_net(coords)  # (N, R, width)
        psi = self.psi_net(coords)  # (N, R, width)

        # s_r = (1/N) * sum_y psi_r(y) . v(y)   -- a Monte-Carlo estimate of the integral.
        s = torch.einsum("nrd,bnd->br", psi, v) / n_points  # (batch, R)

        # u(x) = sum_r phi_r(x) * s_r
        integral_term = torch.einsum("nrd,br->bnd", phi, s)  # (batch, N, width)

        return integral_term + self.local_linear(v)

## The whole network

Same lift -> layers -> project pattern as FNO/GNO. Coordinates are cached per resolution.

In [ ]:
class LNO2d(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, width=32, n_layers=4, rank=16):
        super().__init__()
        self.width = width
        self.lift = nn.Linear(in_channels, width)
        self.layers = nn.ModuleList([LNOLayer(width, rank=rank) for _ in range(n_layers)])
        self.project1 = nn.Linear(width, 128)
        self.project2 = nn.Linear(128, out_channels)
        self._coord_cache = {}

    def _get_coords(self, size_x, size_y, device):
        key = (size_x, size_y, str(device))
        if key not in self._coord_cache:
            gx = torch.linspace(0, 1, size_x, device=device)
            gy = torch.linspace(0, 1, size_y, device=device)
            xx, yy = torch.meshgrid(gx, gy, indexing="ij")
            self._coord_cache[key] = torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=-1)
        return self._coord_cache[key]

    def forward(self, a):
        if a.dim() == 3:
            a = a.unsqueeze(-1)
        batch, size_x, size_y = a.shape[0], a.shape[1], a.shape[2]
        device = a.device
        coords = self._get_coords(size_x, size_y, device)
        n_points = size_x * size_y

        a_flat = a.reshape(batch, n_points, -1)
        coords_b = coords.unsqueeze(0).expand(batch, -1, -1)
        x = torch.cat((a_flat, coords_b), dim=-1)
        v = self.lift(x)

        for t, layer in enumerate(self.layers):
            v = layer(v, coords)
            if t < len(self.layers) - 1:
                v = F.gelu(v)

        out = F.gelu(self.project1(v))
        out = self.project2(out)
        return out.view(batch, size_x, size_y)

## Try it

See how the number of parameters and the output change with the rank $R$ (the saved sweep in `results/lno_summary.json` shows error improving only slowly with rank).

In [ ]:
from neural_operators.utils import count_params

for rank in (2, 4, 8, 16, 32):
    m = LNO2d(rank=rank)
    print(f"rank {rank:>2}: {count_params(m):>8,} parameters")

model = LNO2d()                          # default rank 16; saved result: 309,761 parameters
print("output:", tuple(model(torch.randn(2, 32, 32)).shape))